In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from dataclasses import dataclass
from config import Config
from data_generator import SingleModeDualWavelengthDataGenerator
from model import SimpleMultiWavelengthModel
from trainer import SimpleTrainer
from visualizer import SimpleVisualizer 

def main():
    print("🚀 开始改进的多波长光学神经网络训练")
    
    # 🔥 修复：不要传入field_size=None，让Config自动处理
    config = Config(
        # field_size 参数完全省略，让Config自动检测
        wavelengths=[450e-9, 650e-9],
        offsets=[(-15, 0), (15, 0)],  # 调整偏移距离
        detectsize=10,  # 明确指定检测区域大小
        num_layers=2,   # 增加层数
        epochs=500,     # 增加训练轮数
        learning_rate=1e-3,  # 调整学习率
        batch_size=1
    )
    
    config.print_config()
    
    # 创建数据生成器和模型
    data_generator = SingleModeDualWavelengthDataGenerator(config)
    model = SimpleMultiWavelengthModel(config, num_layers=config.num_layers)
    
    # 创建训练器
    trainer = SimpleTrainer(config, data_generator)
    
    # 训练模型
    result = trainer.train_model(model, num_epochs=config.epochs)
    
    # 可视化结果
    visualizer = SimpleVisualizer(config)
    
    # 生成输入场进行测试
    input_fields = data_generator.generate_input_fields()
    with torch.no_grad():
        output_fields = result['model'](input_fields)
    
    # 绘制结果
    visualizer.plot_energy_distributions(
        output_fields, 
        config.wavelengths, 
        save_path='results/improved_wavelength_separation.png'
    )
    
    # 绘制训练历史
    history = result['training_history']
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 总损失
    axes[0,0].plot(history['total_loss'])
    axes[0,0].set_title('总损失')
    axes[0,0].set_ylabel('损失值')
    axes[0,0].grid(True, alpha=0.3)
    
    # 各项损失
    axes[0,1].plot(history['efficiency_loss'], label='效率损失')
    axes[0,1].plot(history['separation_loss'], label='分离损失')  
    axes[0,1].plot(history['crosstalk_loss'], label='串扰损失')
    axes[0,1].plot(history['concentration_loss'], label='集中损失')
    axes[0,1].set_title('各项损失组成')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 效率变化
    efficiencies = np.array(history['efficiencies'])
    for i, wl in enumerate(config.wavelengths):
        axes[1,0].plot(efficiencies[:, i], label=f'{wl*1e9:.0f}nm')
    axes[1,0].set_title('各波长效率变化')
    axes[1,0].set_ylabel('效率')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 最终效率对比
    final_effs = result['final_efficiencies']
    wl_names = [f'{wl*1e9:.0f}nm' for wl in config.wavelengths]
    axes[1,1].bar(wl_names, final_effs, color=['blue', 'red'])
    axes[1,1].set_title('最终效率对比')
    axes[1,1].set_ylabel('效率')
    for i, eff in enumerate(final_effs):
        axes[1,1].text(i, eff + 0.01, f'{eff:.3f}', ha='center')
    
    plt.tight_layout()
    plt.savefig('results/improved_training_history.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✅ 改进的训练完成!")
    print(f"📈 最终平均效率: {np.mean(final_effs):.3f}")

if __name__ == "__main__":
    main()


🚀 开始改进的多波长光学神经网络训练


TypeError: unsupported operand type(s) for //: 'NoneType' and 'int'